# Step 1.3.a — BIO variables of EC-Earth3 (historical and AMOC runs), raw and delta

Climate of the AMOC scenarios for the SVM: **raw EC-Earth3** (no bias adjustment) through a **delta method on the
BIO variables**, on the EC-Earth3 grid (the SVM grid), land cells of 0.1.

1. **Raw BIO variables** of EC-Earth3 historical (1981–2014) and of the four runs (2025 / 2050 stabilisation, original
   and hosing, 50 years each), with **the same definitions and the same calendar as ERA5**: the variant `CHOSEN_VARIANT`
   (test 1 of 3.g) and the radiation `CHOSEN_RAD` (test 2): `rad_era5` = simulated surface radiation (rsds, needed
   for the runs), `rad_fao56` = estimated from the model's temperatures (FAO-56 eq. 50, as BIO03e / BIO08e of ERA5),
   `no_rad` = none. The calendar (sowing, start of phase 2, maturity) is read from the ERA5 BIO files of 1.1.d, so the
   seasons are identical in every dataset (fixed MIRCA2000 calendar).
2. **Delta BIO variables**, cell by cell, for each run: BIO_scenario = BIO_ERA5 + (BIO_run − BIO_historical) for
   temperatures, radiation, ET0, BIO11, BIO12 (additive change); BIO_scenario = BIO_ERA5 × BIO_run / BIO_historical
   for precipitation (BIO05, BIO10; multiplicative change, the usual delta change for precipitation), additive where
   the historical precipitation is below `PR_MIN` mm/day; precipitation and radiation kept ≥ 0 (not ET0: Hargreaves
   is negative below a mean temperature of −17.8 °C, in ERA5 as well). The level comes
   from ERA5 (1982–2014, the training climate of the SVM), the change from EC-Earth3 (run vs historical 1982–2014).

Outputs: `<out_root>/1_biovariables/GCM_raw_<route>/<variant>_<radiation>/BIOvariables_raw_<dataset>_<crop>.nc`
and `<out_root>/1_biovariables/GCM_delta_<route>/<variant>_<radiation>/BIOvariables_delta_<run>_<crop>.nc`,
with the variable names of the SVM models (3.g), read by 4.g.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the choices of the tests of 3.g
CHOSEN_VARIANT = None              # test 1: "A_60d" | "B_season" | "C_crop"
CHOSEN_RAD = None                  # test 2: "rad_era5" | "no_rad" | "rad_fao56"
CELL_AREA_THR = None              # None = route no_threshold (tests 1-3); 0.0005 = route cellareafrac, presence and
                                  # background = area > 0.05 % of the cell land area (test 4); same value in 1.1.d, 3.g, 1.3.a, 4.g
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")   # ERA5 BIO files of 1.1.d (level of the delta, calendar)
HIST, HIST_PERIOD = "gcm_historical", ("1981", "2014")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]   # whole files (50 years)
PR_MIN = 0.1                        # mm/day: below, the precipitation change is additive instead of a ratio
OVERWRITE = False                   # False = raw BIO files already written are read, not recomputed
assert CHOSEN_VARIANT in ("A_60d", "B_season", "C_crop") and CHOSEN_RAD in ("rad_era5", "no_rad", "rad_fao56"), \
    "set CHOSEN_VARIANT and CHOSEN_RAD to the choices of the tests of 3.g"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
ERA5_TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
RAW = cu.out_dir(CFG, "1_biovariables", f"GCM_raw_{ROUTE}", SET)
DELTA = cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
I, J = np.nonzero(LAND)
LAT_C = GRID["lat"][I]
NC = I.size
lat_rad = np.radians(LAT_C)[:, None]


def era5_file(crop):
    return ERA5_DIR / f"BIOvariables_{ERA5_TAG}_{crop}.nc"


# BIO columns of the chosen set (the SVM models of 3.g use the same), from the ERA5 file
with xr.open_dataset(era5_file(CROPS[0])) as d0:
    base = [v for v in d0.data_vars if v.startswith("BIO") and not v.endswith("e")]
    if CHOSEN_RAD == "rad_era5":
        COLS = base
    elif CHOSEN_RAD == "no_rad":
        COLS = [c for c in base if c not in ("BIO03", "BIO08")]
    else:
        COLS = [c + "e" if c in ("BIO03", "BIO08") else c for c in base]
missing = [c for c in COLS if c not in d0.data_vars]
assert not missing, f"{missing} not in the ERA5 BIO files: run 1.1.d for {CHOSEN_VARIANT} with RAD_EST = True"
PR_COLS = [c for c in COLS if c in ("BIO05", "BIO10")]
print(f"set {SET} | BIO columns {COLS} | land cells {NC}\nraw: {RAW}\ndelta: {DELTA}")

## 1. Functions (ET0 and BIO variables: the code of 1.1.d)

In [ ]:
def compute_Ra(lat_rad_2d, doy):
    Gsc   = 0.0820
    dr    = 1 + 0.033 * np.cos(2 * np.pi / 365 * doy)
    delta = 0.409 * np.sin(2 * np.pi / 365 * doy - 1.39)
    ws    = np.arccos(np.clip(-np.tan(lat_rad_2d) * np.tan(delta), -1, 1))
    return (24*60/np.pi)*Gsc*dr*(ws*np.sin(lat_rad_2d)*np.sin(delta)
                                  + np.cos(lat_rad_2d)*np.cos(delta)*np.sin(ws))

def compute_ETo_day(tmax_day, tmin_day, lat_rad_2d, doy):
    # BUG FIX: Ra from compute_Ra() is in MJ/m2/day (extraterrestrial radiation); the
    # Hargreaves formula needs it as mm/day equivalent (Ra_mm = Ra / lambda). This division
    # was missing here, inflating ETo by a factor of ~lambda (~2.4-2.5x) -- confirmed against
    # AgERA5 Penman-Monteith reference in 1.a)_ET0_method.ipynb. 2_GAEZ's own Hargreaves
    # notebook already does this correctly; this fix makes 1.a consistent with it.
    Tmean = (tmax_day + tmin_day) / 2.0
    TD    = np.maximum(tmax_day - tmin_day, 0.0)
    lam   = 2.501 - 0.002361 * Tmean          # latent heat of vaporization (MJ/kg), temp-dependent
    Ra_mm = compute_Ra(lat_rad_2d, doy) / lam  # MJ/m2/day -> mm/day equivalent
    return 0.0023 * Ra_mm * (Tmean + 17.8) * np.sqrt(TD)

print("ETo functions defined.")

In [ ]:
def compute_bio_variables(TMAX, TMIN, PR, SW, ETO, dates,
                          sowing_doy, anthesis_doy, maturity_doy):
    """
    12 BIO variables averaged over harvest years 1982-2014.
    Calendar days are counted on a 2-year axis (year hy-1 then hy). A season that ends
    in the calendar year of harvest (maturity_doy <= 365) uses year hy; a season that
    crosses New Year (maturity_doy > 365) starts in year hy-1 and ends in year hy.
    Seasons whose start year is missing from the data are skipped for that harvest year
    (each pixel is averaged over the years where it could be computed).
    """
    nlat, nlon = sowing_doy.shape
    yr_all  = dates.year.values
    doy_all = dates.dayofyear.values
    mon_all = dates.month.values
    harvest_years = list(HARVEST_YEARS)
    bio_sum = {i: np.zeros((nlat, nlon)) for i in range(1, 13)}
    bio_cnt = {i: np.zeros((nlat, nlon)) for i in range(1, 13)}
    valid   = ~np.isnan(sowing_doy) & ~np.isnan(maturity_doy)
    wrap    = valid & (maturity_doy > 365)
    yrs_present = set(yr_all)

    def ylen(y): return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365

    for hy in harvest_years:
        print(f'  {hy}...', end=' ')
        if not (yr_all == hy).any(): print('skip'); continue
        idx = np.where((yr_all == hy) | (yr_all == hy - 1))[0]
        yr3  = yr_all[idx][:, None, None]
        doy3 = doy_all[idx][:, None, None]
        mon3 = mon_all[idx][:, None, None]
        t3   = doy3 + np.where(yr3 == hy, ylen(hy - 1), 0)          # day index on the 2-year axis
        shift = np.where(wrap, 0, ylen(hy - 1))[None]                # non-wrapping seasons live in year hy
        sow3 = np.nan_to_num(sowing_doy)[None] + shift
        ant3 = np.nan_to_num(anthesis_doy)[None] + shift
        mat3 = np.nan_to_num(maturity_doy)[None] + shift
        vp3  = valid[None] & (((hy - 1) in yrs_present) | ~wrap[None])

        p1 = (t3 >= sow3) & (t3 < ant3)
        p2 = (t3 >= ant3) & (t3 <= mat3)
        gs = (p1 | p2) & vp3; p1 = p1 & vp3; p2 = p2 & vp3

        tm = TMAX[idx]; tn = TMIN[idx]; pr = PR[idx]; sw = SW[idx]; et = ETO[idx]
        def pm(a, m): return np.nanmean(np.where(m, a, np.nan), axis=0)
        def acc(k, v):
            f = np.isfinite(v); bio_sum[k] += np.where(f, v, 0); bio_cnt[k] += f

        acc(1, pm(tm, p1)); acc(2, pm(tn, p1)); acc(3, pm(sw, p1))
        acc(4, pm(et, p1)); acc(5, pm(pr, p1))
        acc(6, pm(tm, p2)); acc(7, pm(tn, p2)); acc(8, pm(sw, p2))
        acc(9, pm(et, p2)); acc(10, pm(pr, p2))

        tmx = np.full((nlat, nlon), np.nan); tmn = np.full((nlat, nlon), np.nan)
        for m in range(1, 13):
            mm = gs & (mon3 == m)
            if not np.any(mm): continue
            tmx = np.fmax(tmx, np.nanmean(np.where(mm, tm, np.nan), axis=0))
            tmn = np.fmin(tmn, np.nanmean(np.where(mm, tn, np.nan), axis=0))
        acc(11, tmx); acc(12, tmn)
        print('ok')

    bio = {}
    for i in range(1, 13):
        with np.errstate(invalid='ignore', divide='ignore'):
            bio[i] = np.where(bio_cnt[i] > 0, bio_sum[i] / bio_cnt[i], np.nan)
        bio[i][~valid] = np.nan
    return bio

print('compute_bio_variables defined.')

In [ ]:
KEY = {f"BIO{k:02d}": k for k in range(1, 13)}


def calendar(crop):
    # the calendar of the ERA5 BIO file (sowing, start of phase 2, maturity), at the land cells
    with xr.open_dataset(era5_file(crop)) as d:
        return tuple(d[v].values[I, J].astype(float) for v in ("sowing_doy", "phase2_start_doy", "maturity_doy"))


def estimated_radiation(TMAX, TMIN, doys):
    # FAO-56 eq. 50 as in 1.1.d: Rs = kRs sqrt(Tmax - Tmin) Ra, kRs 0.16 inland / 0.19 coastal, capped at 0.75 Ra
    sea = ~LAND
    near_sea = np.zeros_like(sea)
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            near_sea |= np.roll(np.roll(sea, di, axis=0), dj, axis=1)
    krs = np.where((LAND & near_sea)[I, J], 0.19, 0.16).astype(np.float32)
    sw = np.empty_like(TMAX)
    for t in range(doys.size):
        ra = compute_Ra(lat_rad, doys[t])[:, 0]
        sw[t] = np.minimum(krs * np.sqrt(np.maximum(TMAX[t] - TMIN[t], 0.0)) * ra, 0.75 * ra)
    return sw


def raw_bio(name, period):
    # BIO variables (columns COLS) of a GCM dataset for every crop; one file per crop in RAW
    global HARVEST_YEARS
    todo = [c for c in CROPS if OVERWRITE or not (RAW / f"BIOvariables_raw_{name}_{c}.nc").exists()]
    if not todo:
        print(f"[already done] {name}"); return
    t0 = time.time()
    das = {v: cu.read_climate(CFG, name, v, GRID, period) for v in ("tasmax", "tasmin", "pr")}
    dates = pd.DatetimeIndex(das["tasmax"]["time"].values).normalize()
    for v, d in das.items():
        if not np.array_equal(pd.DatetimeIndex(d["time"].values).normalize(), dates):
            raise ValueError(f"{name}: {v} does not have the days of tasmax")
    TMAX, TMIN, PR = (cu.land_series(das[v], (I, J)) for v in ("tasmax", "tasmin", "pr"))
    doys = dates.dayofyear.values
    if CHOSEN_RAD == "rad_era5":
        SW = cu.land_series(cu.read_climate(CFG, name, "rad", GRID, period), (I, J)) * np.float32(0.0864)
    elif CHOSEN_RAD == "rad_fao56":
        SW = estimated_radiation(TMAX, TMIN, doys)
    else:
        SW = np.zeros_like(TMAX)                          # not used (no radiation column)
    ETO = np.empty_like(TMAX)
    for t in range(dates.size):
        ETO[t] = compute_ETo_day(TMAX[t][:, None], TMIN[t][:, None], lat_rad, doys[t])[:, 0]
    years = np.unique(dates.year)
    HARVEST_YEARS = range(int(years[0]) + 1, int(years[-1]) + 1)
    print(f"{name}: {dates[0].date()} -> {dates[-1].date()}, harvest years {HARVEST_YEARS[0]}-{HARVEST_YEARS[-1]}, "
          f"read in {(time.time() - t0) / 60:.1f} min", flush=True)
    for crop in todo:
        sow, ant, mat = calendar(crop)
        bio = compute_bio_variables(TMAX[:, :, None], TMIN[:, :, None], PR[:, :, None], SW[:, :, None], ETO[:, :, None],
                                    dates, sow[:, None], ant[:, None], mat[:, None])
        out = {c: cu_grid(bio[KEY[c.rstrip("e")]][:, 0]) for c in COLS}
        xr.Dataset({c: (("lat", "lon"), x) for c, x in out.items()}, coords={"lat": GRID["lat"], "lon": GRID["lon"]},
                   attrs={"dataset": name, "harvest_years": f"{HARVEST_YEARS[0]}-{HARVEST_YEARS[-1]}", "set": SET,
                          "calendar": str(era5_file(crop).name)}).to_netcdf(RAW / f"BIOvariables_raw_{name}_{crop}.nc")
        print(f"  {crop} done", flush=True)
    print(f"{name}: {(time.time() - t0) / 60:.1f} min")


def cu_grid(x):
    g = np.full(LAND.shape, np.nan, np.float32)
    g[I, J] = x
    return g

## 2. Raw BIO variables: EC-Earth3 historical, then the four runs (one dataset in memory at a time)

In [ ]:
raw_bio(HIST, HIST_PERIOD)
for run in RUNS:
    raw_bio(run, None)

## 3. Delta BIO variables: ERA5 + change of each run relative to EC-Earth3 historical

Additive change for all the variables but precipitation (BIO05, BIO10): ratio, additive where the historical value is
below `PR_MIN`; precipitation and radiation kept ≥ 0. Every land cell (the SVM predicts on all of them).

In [ ]:
NONNEG = [c for c in COLS if c.rstrip("e") in ("BIO03", "BIO05", "BIO08", "BIO10")]   # radiation, precipitation
# (ET0 is not clipped: Hargreaves gives negative values below a mean temperature of -17.8 degC, in ERA5 too)
CRO = cu.mirca_cropland(CFG, CELL_AREA_THR).values
rows = []
for crop in CROPS:
    e5 = xr.open_dataset(era5_file(crop)).load()
    h = xr.open_dataset(RAW / f"BIOvariables_raw_{HIST}_{crop}.nc").load()
    for run in RUNS:
        r = xr.open_dataset(RAW / f"BIOvariables_raw_{run}_{crop}.nc").load()
        out = {}
        for c in COLS:
            if c in PR_COLS:
                ratio = xr.where(h[c] >= PR_MIN, r[c] / h[c], np.nan)
                v = xr.where(h[c] >= PR_MIN, e5[c] * ratio, e5[c] + (r[c] - h[c]))
            else:
                v = e5[c] + (r[c] - h[c])
            out[c] = v.clip(min=0) if c in NONNEG else v
        ds = xr.Dataset(out).astype("float32")
        for v in ("sowing_doy", "phase2_start_doy", "maturity_doy"):
            ds[v] = e5[v]
        ds.attrs.update({"description": f"delta BIO variables: ERA5 {ERA5_PERIOD[0]}-{ERA5_PERIOD[1]} + EC-Earth3 change "
                                        f"({run} - {HIST}), {SET}, crop={crop}",
                         "precipitation": f"ratio (additive where {HIST} < {PR_MIN} mm/day)", "others": "additive"})
        ds.to_netcdf(DELTA / f"BIOvariables_delta_{run}_{crop}.nc")
        row = {"crop": crop, "run": run}
        for c in COLS:
            if c in PR_COLS:
                ok = CRO & (h[c].values >= PR_MIN)
                row[f"{c} ratio"] = float(np.nanmean((r[c] / h[c]).values[ok]))
            else:
                row[f"{c} change"] = float(np.nanmean((r[c] - h[c]).values[CRO]))
        row["cells with additive pr change"] = int(sum(int((h[c].values[LAND] < PR_MIN).sum()) for c in PR_COLS))
        rows.append(row)
    print(f"{crop}: delta files written", flush=True)
changes = pd.DataFrame(rows).set_index(["crop", "run"])
changes.to_csv(DELTA / f"changes_on_cropland_{SET}.csv")
print("Mean change of each BIO variable over the MIRCA2000 cropland (run - historical; precipitation: ratio)")
with pd.option_context("display.max_columns", 30, "display.width", 250):
    display(changes.round(2))

## 4. Check: maps of BIO01 and BIO05 (or BIO10) for one crop, ERA5 and the four delta scenarios

In [ ]:
crop = CROPS[0]
show = [c for c in ("BIO01", PR_COLS[0] if PR_COLS else None) if c]
fig, axes = plt.subplots(len(show), 1 + len(RUNS), figsize=(5 * (1 + len(RUNS)), 3 * len(show)), squeeze=False)
e5 = xr.open_dataset(era5_file(crop))
for i, c in enumerate(show):
    vmin, vmax = float(np.nanpercentile(e5[c], 2)), float(np.nanpercentile(e5[c], 98))
    e5[c].plot(ax=axes[i, 0], vmin=vmin, vmax=vmax, add_colorbar=False); axes[i, 0].set_title(f"ERA5 {c}", fontsize=9)
    for j, run in enumerate(RUNS, 1):
        d = xr.open_dataset(DELTA / f"BIOvariables_delta_{run}_{crop}.nc")
        d[c].plot(ax=axes[i, j], vmin=vmin, vmax=vmax, add_colorbar=False); axes[i, j].set_title(f"{run} {c}", fontsize=9)
for ax in axes.ravel():
    ax.set_xlabel(""); ax.set_ylabel("")
plt.suptitle(f"{crop} ({SET})"); plt.tight_layout()
plt.savefig(DELTA / f"check_delta_{SET}_{crop}.png", dpi=80); plt.show()